# M2.6 · PCA ve SVD

Verinin en çok değiştiği yönler. 2×2 kovaryansın özdeğerlerini ve power iteration'ın bir adımını elle hesaplıyor, saf Python'da deflation ile ilk k bileşeni buluyor, NumPy'da SVD ile PCA ve yeniden kurma yapıyor, PyTorch'ta baş bileşeni Rayleigh bölümünü autograd ile maksimize ederek buluyor ve en sonda yüzlerce küçük veri setinin PCA'sını tek batched SVD ile alıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m2-6-pca.html](https://m-gonul.github.io/pytorch-lab/m2-6-pca.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m2_6_pca.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef sign_fix(V):\n    idx = np.argmax(np.abs(V), axis=1)\n    return V * np.sign(V[np.arange(len(V)), idx])[:, None]\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    rng = np.random.default_rng(0)\n    X = rng.normal(size=(40, 3)) @ np.array([[2.0, 0.5, 0], [0, 1.0, 0.3], [0, 0, 0.4]])\n\n    def cov():\n        close(need(ns, \"covariance\")(X.tolist()), np.cov(X.T))\n    R.test(\"covariance = np.cov\", cov)\n\n    def pw():\n        lam, v = need(ns, \"power_iteration\")([[2.0, 1.0], [1.0, 2.0]])\n        close([lam, v[0], v[1]], [3, 0.5 ** 0.5, 0.5 ** 0.5])\n        lam2, v2 = need(ns, \"power_iteration\")([[1.0, 0], [0, -4.0]])\n        close([lam2, v2[0], v2[1]], [-4, 0, 1], atol=1e-6, what=\"negatif özdeğer: |λ| en büyük, işaret kuralı\")\n    R.test(\"power_iteration: elle örnek ve işaret kuralı\", pw)\n\n    def tk():\n        C = np.cov(X.T)\n        lams, vecs = need(ns, \"top_k\")(C.tolist(), 3, 500)\n        w, V = np.linalg.eigh(C)\n        close(lams, w[::-1], rtol=1e-6, what=\"özdeğerler\")\n        close(np.array(vecs), sign_fix(V[:, ::-1].T), atol=1e-5, rtol=0, what=\"özvektörler\")\n    R.test(\"top_k (deflation) = eigh\", tk)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef sign_fix(V):\n    idx = np.argmax(np.abs(V), axis=1)\n    return V * np.sign(V[np.arange(len(V)), idx])[:, None]\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(1)\n    X = rng.normal(size=(150, 4)) @ rng.normal(size=(4, 4)) + np.array([10.0, -3, 0, 7])\n    w, V = np.linalg.eigh(np.cov(X.T))\n\n    def comp():\n        P, var, Z = need(ns, \"pca\")(X, 3)\n        ensure(np.asarray(P).shape == (3, 4) and np.asarray(Z).shape == (150, 3), \"shape'ler\")\n        close(P @ P.T, np.eye(3), atol=1e-9, what=\"ortonormallik\")\n        close(var, w[::-1][:3], rtol=1e-8, what=\"varyanslar = özdeğerler\")\n        close(P, sign_fix(V[:, ::-1].T)[:3], atol=1e-8, what=\"bileşenler (işaret kuralıyla)\")\n        close(Z.var(0, ddof=1), w[::-1][:3], rtol=1e-8, what=\"Z sütun varyansları\")\n    R.test(\"bileşenler, varyanslar, işaret kuralı\", comp)\n\n    def rec():\n        P, var, Z = need(ns, \"pca\")(X, 4)\n        close(need(ns, \"reconstruct\")(Z, P, X.mean(0)), X, atol=1e-9, what=\"k = d tam kurma\")\n        P2, _, Z2 = need(ns, \"pca\")(X, 2)\n        err = ((need(ns, \"reconstruct\")(Z2, P2, X.mean(0)) - X) ** 2).sum() / (len(X) - 1)\n        close(err, w[::-1][2:].sum(), rtol=1e-8, what=\"k = 2 hatası = atılan özdeğerlerin toplamı\")\n        return f\"k = 2 yeniden kurma hatası {err:.4f} = λ₃ + λ₄\"\n    R.test(\"yeniden kurma: kayıp = atılan varyans\", rec)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    g = torch.Generator().manual_seed(0)\n    X = torch.randn(300, 4, generator=g, dtype=torch.float64) @ torch.randn(4, 4, generator=g, dtype=torch.float64)\n    Xc = X - X.mean(0)\n    U, s, Vh = torch.linalg.svd(Xc, full_matrices=False)\n    idx = Vh.abs().argmax(1, keepdim=True)\n    Vref = Vh * Vh.gather(1, idx).sign()\n\n    def pt():\n        P, var = need(ns, \"pca_torch\")(X, 2)\n        close(var, s[:2] ** 2 / 299, rtol=1e-8); close(P, Vref[:2], atol=1e-8)\n    R.test(\"pca_torch = SVD (işaret kuralıyla)\", pt)\n    R.test(\"autograd kullanılıyor\", lambda: ensure(uses_any(src, [\"backward\", \"grad\"]), \"Rayleigh bölümünü autograd ile optimize et\"))\n\n    def gd():\n        v = need(ns, \"top_component_gd\")(X)\n        ensure(abs(float(v.norm()) - 1) < 1e-6, \"birim vektör bekleniyordu\")\n        cos = float(v @ Vref[0])\n        ensure(cos > 0.9999, f\"baş bileşenle kosinüs {cos:.6f}\")\n        return f\"cos(v, v₁) = {cos:.7f}\"\n    R.test(\"top_component_gd baş bileşene yakınsıyor\", gd)\n",
 "e": "import torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef one(X, k):\n    Xc = X - X.mean(0)\n    U, s, Vh = torch.linalg.svd(Xc, full_matrices=False)\n    V = Vh[:k]\n    idx = V.abs().argmax(1, keepdim=True)\n    return V * V.gather(1, idx).sign(), s[:k] ** 2 / (X.shape[0] - 1)\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    Xs = torch.randn(16, 40, 5, generator=g, dtype=torch.float64) * torch.tensor([4.0, 2, 1, 0.5, 0.1], dtype=torch.float64)\n\n    def eq():\n        P, var = need(ns, \"batched_pca\")(Xs, 3)\n        ensure(tuple(P.shape) == (16, 3, 5) and tuple(var.shape) == (16, 3), f\"shape'ler {tuple(P.shape)}, {tuple(var.shape)}\")\n        for b in range(16):\n            Pb, vb = one(Xs[b], 3)\n            close(P[b], Pb, atol=1e-9, what=f\"veri seti {b} bileşenler\"); close(var[b], vb, what=f\"veri seti {b} varyanslar\")\n    R.test(\"16 veri seti: tek tek PCA ile aynı\", eq)\n    if \"batched_pca\" not in ns:\n        return\n    Xb = torch.randn(1000, 32, 8, generator=g)\n    t_loop = bench(lambda: [one(x, 3) for x in Xb], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"batched_pca\"](Xb, 3), repeat=3)\n    R.table(\"1000 küçük PCA (32 × 8)\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for: 1000 ayrı SVD\", fmt_time(t_loop), \"1×\"], [\"senin batched_pca\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]])\n"
}

In [ ]:
import importlib, labcheck, checks_m2_6_pca as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Ortalanmış veri $X_c\in\mathbb R^{n\times d}$ ve kovaryansı $C=\frac1{n-1}X_c^\top X_c$. Birim bir yön $v$ boyunca izdüşümlerin varyansı $v^\top Cv$'dir. Bunu maksimize eden yön $C$'nin en büyük özdeğerine ait özvektördür:

$$
\max_{\lVert v\rVert=1}v^\top Cv\quad\Rightarrow\quad Cv=\lambda v .
$$

İlk $k$ özvektör (bileşenler) veriyi $k$ boyuta en az bilgi kaybıyla indirir: $Z=X_cV_k$, yeniden kurma $\hat X=ZV_k^\top+\bar x$.

Hesaplamanın iki yolu:

- **Power iteration**: $v\leftarrow Cv/\lVert Cv\rVert$ tekrarı en büyük özvektöre yakınsar (hız $\lambda_2/\lambda_1$ oranına bağlı). Sonraki bileşen için **deflation**: $C\leftarrow C-\lambda vv^\top$.
- **SVD**: $X_c=USV^\top$ ise $V$'nin sütunları bileşenler, $\lambda_i=s_i^2/(n-1)$.

İşaret kuralı (her aşamada): her bileşende mutlak değerce en büyük eleman pozitif olacak şekilde gerekirse $-1$ ile çarp.

## 1 · Elle

Önce özdeğerleri karakteristik polinomdan bul, sonra $(C-3I)v=0$'ı çöz. Toplam varyans (iz) $2+2=4$; ilk bileşen bunun $\%75$'ini taşıyor.

Power iteration'ı bir adım uygula ve Rayleigh bölümünü hesapla: tek adımda 2.8'e çıkıyor (gerçek 3). Bir adım daha atarsan $v\approx(0.78,0.62)$ ve bölüm $2.98$ olur. Yakınsama hızı $\lambda_2/\lambda_1=1/3$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Kovaryans $C=\begin{bmatrix}2&1\\1&2\end{bmatrix}$; $\det(C-\lambda I)=(2-\lambda)^2-1=0$**

- `l1`: $\lambda_1$ (büyük)
- `l2`: $\lambda_2$
- `v1x`: $v_1$'in 1. bileşeni (birim, pozitif)
- `ratio`: 1. bileşenin açıkladığı varyans oranı
- `proj`: $x=(2,0)$'ın $v_1$ üzerine izdüşümü $x\cdot v_1$

**Power iteration $v\leftarrow Cv/\lVert Cv\rVert$, $v_0=(1,0)$**

- `p1x`: $v_1$'in 1. bileşeni (bir adım)
- `p1y`: … 2. bileşeni
- `rq`: Rayleigh $v^\top Cv$ (bir adım sonra)

In [ ]:
cevaplar = dict(
    l1=None,
    l2=None,
    v1x=None,
    ratio=None,
    proj=None,
    p1x=None,
    p1y=None,
    rq=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"l1":[3,0],"l2":[1,0],"v1x":[0.707107,0.002],"ratio":[0.75,0.001],"proj":[1.414214,0.002],"p1x":[0.894427,0.002],"p1y":[0.447214,0.002],"rq":[2.8,0.002]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python. `X` liste listesi.

- `covariance(X)` — `(d × d)`, payda $n-1$.
- `power_iteration(C, iters=200)` → `(λ, v)`; $v_0=(1,1,…,1)/\sqrt d$, normalize, sonunda işaret kuralı, $\lambda=v^\top Cv$.
- `top_k(C, k, iters=200)` → `(lambdas, vectors)`; her bileşenden sonra deflation.

`numpy`/`torch` yasak.

In [ ]:
%%writefile stage_b.py
import math


def covariance(X):
    ...


def power_iteration(C, iters=200):
    ...


def top_k(C, k, iters=200):
    ...


print(power_iteration([[2.0, 1.0], [1.0, 2.0]]))   # (3.0, [0.7071, 0.7071])
print(top_k([[2.0, 1.0], [1.0, 2.0]], 2))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

- `pca(X, k)` → `(components: (k, d), variances: (k,), Z: (n, k))`. Ortala, `np.linalg.svd(Xc, full_matrices=False)`, ilk $k$ satır `Vt`, varyans $s^2/(n-1)$, işaret kuralını vektörel uygula (`np.argmax(np.abs(V), axis=1)` ile her satırın en büyük elemanı). $Z=X_cV^\top$.
- `reconstruct(Z, components, mean)` → `(n, d)`.

**Döngü yok.** Kontrol: bileşenler ortonormal, varyanslar kovaryansın özdeğerleriyle aynı, tam $k=d$'de yeniden kurma kayıpsız.

In [ ]:
%%writefile stage_c.py
import numpy as np


def pca(X, k):
    """→ (components (k, d), variances (k,), Z (n, k)). Döngü yok."""
    ...


def reconstruct(Z, components, mean):
    ...


rng = np.random.default_rng(0)
X = rng.normal(size=(200, 3)) @ np.array([[3.0, 1.0, 0], [0, 1.0, 0], [0, 0, 0.1]]) + 5
comps, var, Z = pca(X, 2)
print(comps, var)
print(np.abs(reconstruct(Z, comps, X.mean(0)) - X).max())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Baş bileşeni bir optimizasyon problemi olarak çöz: Rayleigh bölümü $R(v)=\frac{v^\top Cv}{v^\top v}$ ölçekten bağımsız, maksimumu $\lambda_1$.

- `pca_torch(X, k)` → `(components, variances)`; `torch.linalg.eigh` (artan sırada döner; ters çevir) + işaret kuralı.
- `top_component_gd(X, steps=500, lr=0.05)` → birim `v`; $-R(v)$'yi `Adam` ile minimize et (autograd), sonunda normalize + işaret kuralı.

In [ ]:
%%writefile stage_d.py
import torch


def pca_torch(X, k):
    """→ (components (k, d), variances (k,)); eigh ile."""
    ...


def top_component_gd(X, steps=500, lr=0.05):
    """Rayleigh bölümünü autograd ile maksimize et → birim v."""
    ...


torch.manual_seed(0)
X = torch.randn(300, 3) @ torch.tensor([[3.0, 1.0, 0], [0, 1.0, 0], [0, 0, 0.2]])
print(pca_torch(X, 1)[0], top_component_gd(X))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Yüzlerce küçük veri setinin (ör. her görüntü yaması, her zaman penceresi) PCA'sı: döngü yerine **batched SVD**. `torch.linalg.svd` baştaki boyutları batch sayar.

- `batched_pca(Xs, k)` — `Xs: (B, n, d)` → `(components: (B, k, d), variances: (B, k))`. Ortalama `Xs.mean(1, keepdim=True)`. İşaret kuralı vektörel: `idx = Vk.abs().argmax(-1, keepdim=True)`, `sign = Vk.gather(-1, idx).sign()`. **Döngü yok.**

In [ ]:
%%writefile stage_e.py
import torch


def batched_pca(Xs, k):
    """Xs: (B, n, d) → (components (B, k, d), variances (B, k)). Döngü yok."""
    ...


Xs = torch.randn(4, 50, 3) * torch.tensor([3.0, 1.0, 0.3])
P, var = batched_pca(Xs, 2)
print(P.shape, var)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Halko, Martinsson, Tropp (2011): rastgele matris ayrışımları

[Halko, Martinsson, Tropp — Finding Structure with Randomness: Probabilistic Algorithms for Constructing Approximate Matrix Decompositions (SIAM Review 2011)](https://arxiv.org/abs/0909.4061)

Rastgele SVD'yi (Algoritma 4.1 + 5.1) yaz: Gauss test matrisi, aşırı örnekleme p ve q güç yinelemesi. Tekil değerleri yavaş ve hızlı sönen matrislerde (ör. görüntü yamaları, sentetik spektrumlar) rank-k yaklaşım hatasını kuramsal sınırla (Bölüm 10) ve np.linalg.svd ile karşılaştır; süre ölçeklemesini büyük matrislerde ölç. sklearn PCA(svd_solver='randomized') ile aynı sonucu verdiğini göster.


**Adımlar**

1. Güç yinelemesinde her adımda QR ile yeniden ortogonalleştir.

**Kabul ölçütleri**

- Hata kesin SVD'nin σ_{k+1}'ine yakın; q arttıkça iyileşir.
- Büyük matrislerde belirgin hızlanma.

## Kaynaklar

- [Mathematics for Machine Learning — Bölüm 10 (PCA)](https://mml-book.github.io/) — © (PDF ücretsiz)
- [MIT 18.06 — Singular Value Decomposition (Strang)](https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/) — CC BY-NC-SA 4.0
- [ISLP — §12.2 Principal Components Analysis](https://www.statlearning.com/) — doğrulanmadı
- [PyTorch — torch.linalg.svd / eigh (batched)](https://docs.pytorch.org/docs/stable/generated/torch.linalg.svd.html) — BSD-3